In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 5: Fix placeholder past-position values
# =============================================================================
# Step:         5 of 7
# Summary:      Replace the 999999 SUMO sentinel in lagged position columns with each vehicle's first known coordinate.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.1.3
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 5 — Fix placeholder past-position values

Part of the **FUMD-AI preprocessing workflow** (step 5 of 7).

**Purpose.** The SUMO trajectory file encodes "no data yet" for a vehicle's
lagged position columns (`x-1..x-7`, `y-1..y-7`) using the sentinel value
`999999`, for vehicles that hadn't been driving long enough yet to have
N seconds of history (e.g. a vehicle 2 seconds into its trip has no real
`x-5` value). Left as-is, `999999` would be read by a model as a real,
enormous coordinate. This notebook replaces every sentinel with that
vehicle's own first known coordinate instead - a reasonable stand-in for
"as far back as we know, the vehicle was already here".

**Input:** `INPUT_PATH` - the merged dataset from Step 4.

**Output:** `OUTPUT_PATH` - same dataset, with sentinel values replaced.


In [ ]:
import pandas as pd


In [ ]:
# ---- Parameters ----
# Defaults chain onto Step 4's default COMBINED_PATH, so this notebook runs
# out of the box straight after Step 4 using the bundled example.
INPUT_PATH = "combined_dataset.csv"          # Step 4 output (input)
OUTPUT_PATH = "combined_dataset_fixed.csv"   # sentinel-fixed output -> feeds Step 6's INPUT_PATH
SENTINEL = 999999          # placeholder value used by SUMO for "no data yet"
N_LAG = 7                  # number of lagged x-N/y-N column pairs to fix


## 1. Load and confirm the sentinel is present

In [ ]:
data = pd.read_csv(INPUT_PATH)
data["veh_id"] = data["veh_id"].astype(int)  # guard against a float dtype from upstream merges

contains_sentinel = (data == SENTINEL).any().any()
print(f"dataset contains {SENTINEL} placeholders: {contains_sentinel}")
data.head()


## 2. Get each vehicle's first known coordinate

In [ ]:
# sort by time so .first() (per vehicle) really means "earliest sample",
# then keep only the columns we need and rename to avoid clashing with the
# existing "x"/"y" columns once merged back in below
first_seen = (
    data.sort_values("t")
    .groupby("veh_id", as_index=False)
    .first()[["veh_id", "x", "y"]]
    .rename(columns={"x": "firstx", "y": "firsty"})
)
# attach each vehicle's first-known coordinate onto every one of its rows
data = data.merge(first_seen, on="veh_id")
first_seen.head()


## 3. Replace the sentinel value in the lagged position columns

In [ ]:
# for every lag 1..N_LAG, wherever that column still holds the sentinel,
# overwrite it with the vehicle's first-known (x, y) - i.e. "as far back
# as we know, the vehicle was already at its starting position"
for lag in range(1, N_LAG + 1):
    xcol, ycol = f"x-{lag}", f"y-{lag}"
    data.loc[data[xcol] == SENTINEL, xcol] = data["firstx"]
    data.loc[data[ycol] == SENTINEL, ycol] = data["firsty"]

still_contains_sentinel = (data == SENTINEL).any().any()
print(f"dataset still contains {SENTINEL} placeholders: {still_contains_sentinel}")
assert not still_contains_sentinel, "sentinel values remain - check N_LAG / column names"


## 4. Drop helper columns and save

In [ ]:
# firstx/firsty were only needed to do the replacement above
data = data.drop(columns=["firstx", "firsty"])
data.to_csv(OUTPUT_PATH, index=False)
print(f"saved {OUTPUT_PATH} - shape {data.shape}")
data.head()
